This notebook uses the [Meta Kaggle](https://www.kaggle.com/kaggle/meta-kaggle) dataset and computes a positive/negative sentiment score for all the Kaggle forum posts using the [HuggingFace Sentiment Analysis Pipeline](https://huggingface.co/transformers/main_classes/pipelines.html), saving the results for later analysis.

___

See the follow-up Notebook here: [A Look at Sentiment on the Kaggle Forums](https://www.kaggle.com/jtrotman/a-look-at-sentiment-on-the-kaggle-forums)

In [1]:
from jt_mk_utils import *
from transformers import pipeline
from bs4 import BeautifulSoup
import re
import numpy as np
from tqdm.notebook import tqdm

In [2]:
classifier = pipeline("sentiment-analysis",
                      model="distilbert-base-uncased-finetuned-sst-2-english",
                      framework="pt",
                      device=0)
# max tokens evaluated per forum post: long posts are truncated
classifier.tokenizer.model_max_length

In [3]:
def get_html_text(s):
    soup = BeautifulSoup(s, "html.parser")
    # removed quoted text written by others
    for data in soup(["style", "script", "iframe", "blockquote", "code"]):
        data.decompose()
    txt = soup.get_text()
    txt = re.sub(r"\[quote.*\[/quote\]", " ", txt, flags=re.S)
    txt = txt.strip()
    return txt

In [4]:
forum_messages = read_forum_messages(index_col=0) # can add this arg to test: nrows=50_000)
forum_messages["Message"] = forum_messages["Message"].fillna("")
forum_messages.shape

# Parse HTML

In [5]:
missing_para = ~forum_messages["Message"].str.lower().str.startswith("<p>")
missing_para.mean()

In [6]:
forum_messages.loc[missing_para, "Message"] = ("<p>") + forum_messages.loc[missing_para, "Message"]

In [7]:
%%time
text = forum_messages["Message"].apply(get_html_text)

In [8]:
# sort by length, batches with lower maximum length are quicker
text = text.iloc[text.str.len().argsort()]
text.shape

# Run Pipeline

In [9]:
%%time
res = classifier(list(text), batch_size=64, truncation=True)

In [10]:
df = pd.DataFrame(res, index=text.index)
df.shape

In [11]:
df.label.value_counts()

In [12]:
df.groupby("label").score.agg(["count", "sum", "min", "mean", "max"])

# Save

In [13]:
df.sort_index().to_csv("forum-messages-sentiment-analysis.csv")

Check out [A Look at Sentiment on the Kaggle Forums](https://www.kaggle.com/jtrotman/a-look-at-sentiment-on-the-kaggle-forums) to see some ways to use these scores.